# ⚽ 월문FC 하이라이트 자동 편집기

**사용 방법 (모바일):**
1. 위 메뉴 `런타임 → 모두 실행` 탭
2. **셀 2** 에서 영상 파일 업로드
3. 기다리면 자동으로 하이라이트 다운로드 시작

---
- 오디오 에너지(군중 함성) + 모션(빠른 움직임) 분석으로 자동 감지
- 출력: 1920×1080 H.264 유튜브 최적화 MP4

In [ ]:
# ── 셀 1: 패키지 설치 (자동) ──────────────────────────────────
import subprocess, sys
print('🔧 ffmpeg 확인 중...')
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'opencv-python-headless', 'scipy', 'numpy'], capture_output=True)
print('✅ 준비 완료!')

In [ ]:
# ── 셀 2: 영상 업로드 ──────────────────────────────────────────
from google.colab import files
print('📱 경기 영상을 선택하세요 (MP4, MOV, AVI 등)')
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f'\n✅ 업로드 완료: {video_path}')

In [ ]:
# ── 셀 3: 설정 ────────────────────────────────────────────────

# 하이라이트 목표 길이 (초)
# 90  = 1분 30초 (핵심만)
# 180 = 3분      (기본값)
# 300 = 5분
TARGET_SECONDS = 180

print(f'🎯 하이라이트 목표 길이: {TARGET_SECONDS // 60}분 {TARGET_SECONDS % 60}초')

In [ ]:
# ── 셀 4: 하이라이트 감지 + 편집 (자동) ───────────────────────
import re, json, os, shutil
import numpy as np
from pathlib import Path

def get_video_info(path):
    r = subprocess.run(
        ['ffprobe', '-v', 'quiet', '-print_format', 'json',
         '-show_streams', '-show_format', str(path)],
        capture_output=True, text=True, check=True)
    d = json.loads(r.stdout)
    duration = float(d['format']['duration'])
    fps = 30.0
    for s in d['streams']:
        if s.get('codec_type') == 'video':
            try:
                n, den = s.get('r_frame_rate','30/1').split('/')
                fps = float(n) / float(den)
            except: pass
    return duration, fps

def analyze_audio(path):
    r = subprocess.run(
        ['ffmpeg', '-i', str(path), '-af', 'ebur128=framelog=verbose', '-f', 'null', '-'],
        capture_output=True, text=True)
    times, vals = [], []
    for line in r.stderr.split('\n'):
        m = re.search(r't:\s*([\d.]+)\s+M:\s*([-\d.inf]+)', line)
        if m:
            t = float(m.group(1))
            v = -70.0 if 'inf' in m.group(2) else float(m.group(2))
            times.append(t); vals.append(max(v, -70.0))
    return np.array(times), np.array(vals)

def analyze_motion(path, sample_fps=2):
    import cv2
    cap = cv2.VideoCapture(str(path))
    vfps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(1, int(vfps / sample_fps))
    times, scores = [], []
    prev = None; idx = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if idx % step == 0:
            t = idx / vfps
            g = cv2.cvtColor(cv2.resize(frame, (320,180)), cv2.COLOR_BGR2GRAY)
            if prev is not None:
                scores.append(float(np.mean(cv2.absdiff(g, prev))))
                times.append(t)
            prev = g
        idx += 1
    cap.release()
    return np.array(times), np.array(scores)

def normalize(a):
    mn, mx = a.min(), a.max()
    return np.zeros_like(a) if mx - mn < 1e-6 else (a - mn) / (mx - mn)

def compute_scores(dur, at, av, mt, mv):
    from scipy.ndimage import gaussian_filter1d
    n = int(dur) + 1
    tg = np.arange(n, dtype=float)
    ag = np.interp(tg, at, normalize(av)) if len(at) > 1 else np.zeros(n)
    mg = np.interp(tg, mt, normalize(mv)) if len(mt) > 1 else np.zeros(n)
    return tg, gaussian_filter1d(0.6*ag + 0.4*mg, sigma=5)

def select_segments(times, scores, target, before=15, after=20, gap=8):
    dur = float(times[-1])
    work = scores.copy(); segs = []; total = 0.0
    while total < target and work.max() > 0.05:
        idx = int(np.argmax(work))
        pt = float(times[idx])
        s, e = max(0.0, pt-before), min(dur, pt+after)
        segs.append([s, e]); total += e - s
        work[(times >= max(0,pt-before-gap)) & (times <= min(dur,pt+after+gap))] = 0
    segs.sort(key=lambda x: x[0])
    merged = []
    for seg in segs:
        if merged and seg[0] <= merged[-1][1] + gap:
            merged[-1][1] = max(merged[-1][1], seg[1])
        else:
            merged.append(seg)
    return [(round(s,1), round(e,1)) for s,e in merged]

def render(video_path, segments, output_path):
    tmpdir = Path('_tmp_clips'); tmpdir.mkdir(exist_ok=True)
    clips = []
    for i, (s, e) in enumerate(segments):
        cp = tmpdir / f'clip_{i:03d}.mp4'
        subprocess.run([
            'ffmpeg', '-y', '-ss', str(s), '-to', str(e), '-i', str(video_path),
            '-c:v', 'libx264', '-crf', '23', '-preset', 'fast',
            '-c:a', 'aac', '-b:a', '128k',
            '-vf', 'scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2:black',
            '-movflags', '+faststart', str(cp)
        ], capture_output=True, check=True)
        clips.append(cp)
        print(f'  클립 {i+1}/{len(segments)} 완료 ({s:.0f}s ~ {e:.0f}s)')
    concat = tmpdir / 'list.txt'
    concat.write_text('\n'.join(f"file '{p.resolve()}'" for p in clips))
    subprocess.run([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
        '-i', str(concat), '-c', 'copy', '-movflags', '+faststart', str(output_path)
    ], capture_output=True, check=True)
    shutil.rmtree(tmpdir, ignore_errors=True)

# ── 실행 ──
print('🎬 영상 정보 분석 중...')
duration, fps = get_video_info(video_path)
print(f'   길이: {duration//60:.0f}분 {duration%60:.0f}초  FPS: {fps:.1f}')

print('🔊 오디오 에너지 분석 중... (군중 소리 감지)')
at, av = analyze_audio(video_path)

print('🏃 모션 분석 중... (빠른 움직임 감지)')
mt, mv = analyze_motion(video_path)

print('🎯 하이라이트 구간 선정 중...')
tg, scores = compute_scores(duration, at, av, mt, mv)
segments = select_segments(tg, scores, TARGET_SECONDS)

print(f'\n✅ {len(segments)}개 구간 선택됨:')
for s, e in segments:
    m1,s1 = divmod(int(s), 60)
    m2,s2 = divmod(int(e), 60)
    print(f'   {m1}:{s1:02d} ~ {m2}:{s2:02d}')

print('\n✂️  클립 렌더링 중 (유튜브용 1920×1080)...')
output = Path(video_path).stem + '_highlight.mp4'
render(video_path, segments, output)
print(f'\n🎉 완성! → {output}')

In [ ]:
# ── 셀 5: 다운로드 ────────────────────────────────────────────
from google.colab import files
print('⬇️  다운로드 시작...')
files.download(output)
print('✅ 완료! 유튜브에 바로 업로드하세요.')